# 🧠 LOGOS — Vedic-Physics Hybrid LLM Training on Kaggle

In [ ]:
# STEP 1 — Env check
import os, glob, subprocess, sys, re, math, shutil
os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
os.system('g++ --version | head -1')
os.system('cmake --version | head -1')
os.system('nproc')

In [ ]:
# STEP 2 — train.bin → dataset.txt
os.system('pip install tiktoken -q')
import tiktoken, numpy as np

BIN_FILE  = '/kaggle/input/datasets/josephmayok/roneneldan-tinystories/train.bin'
TRAIN_TXT = '/kaggle/working/dataset.txt'

enc  = tiktoken.get_encoding('gpt2')
data = np.fromfile(BIN_FILE, dtype=np.uint16)
print(f'Tokens in bin: {len(data):,}')

CHUNK = 100_000
with open(TRAIN_TXT, 'w', encoding='utf-8') as f:
    for i in range(0, len(data), CHUNK):
        f.write(enc.decode(data[i:i+CHUNK].tolist()))
        if i % (CHUNK*10) == 0:
            print(f'  {i:,} tokens decoded...')

mb = os.path.getsize(TRAIN_TXT)/1024/1024
print(f'\n✅ dataset.txt: {mb:.1f} MB')
with open(TRAIN_TXT) as f: print('\nPreview:', f.read(200))

In [ ]:
# STEP 3 — Build LOGOS
WORK_DIR = '/kaggle/working/LOGOS'
os.chdir(WORK_DIR)

# Pehle purana build clean karo
os.system('rm -rf build')

ret = os.system('''
    cmake -B build \
        -DCMAKE_BUILD_TYPE=Release \
        -DCMAKE_CXX_FLAGS="-O3 -march=native -std=c++20" \
    && cmake --build build --parallel $(nproc) 2>&1
''')

if ret == 0:
    print('\n✅ BUILD SUCCESS')
    os.system('ls -lh build/logos')
else:
    print('\n❌ BUILD FAILED — error dekho upar')

In [ ]:
# STEP 4 — Tests
os.chdir(WORK_DIR)
os.system('./build/logos --test')
os.system('./build/logos --forward')
os.system('./build/logos --benchmark')

In [ ]:
# STEP 5 — 🚀 TRAIN
os.chdir(WORK_DIR)
TRAIN_FILE = '/kaggle/working/dataset.txt'
LOG_FILE   = '/kaggle/working/training_log.txt'

print('🚀 LOGOS Training START')
print(f'   Dataset: {os.path.getsize(TRAIN_FILE)//1024//1024} MB')
print('   Loss ~8.3 se shuru hoga, girte dekhna...\n')

steps_log, losses_log = [], []

process = subprocess.Popen(
    ['./build/logos', '--train', TRAIN_FILE],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
    cwd=WORK_DIR
)

with open(LOG_FILE, 'w') as log:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
            m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
            if m:
                steps_log.append(int(m.group(1)))
                losses_log.append(float(m.group(2)))
    except KeyboardInterrupt:
        process.terminate()
        print('\n⏹️  Stopped')

process.wait()
if losses_log:
    print(f'\nStart : {losses_log[0]:.4f}')
    print(f'Final : {losses_log[-1]:.4f}')
    print(f'Drop  : {losses_log[0]-losses_log[-1]:.4f} ✅')

In [ ]:
# STEP 6 — Loss Curve Plot
import matplotlib.pyplot as plt

steps_log, losses_log = [], []
with open('/kaggle/working/training_log.txt') as f:
    for line in f:
        m = re.search(r'Step\s+(\d+).*Loss:\s+([\d.]+)', line)
        if m:
            steps_log.append(int(m.group(1)))
            losses_log.append(float(m.group(2)))

if steps_log:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
    ax1.plot(steps_log, losses_log, 'b-', lw=1.5)
    ax1.axhline(y=losses_log[-1], color='r', ls='--',
                label=f'Final: {losses_log[-1]:.3f}')
    ax1.set_title('LOGOS Loss (Langevin Optimizer)')
    ax1.set_xlabel('Steps'); ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3); ax1.legend()

    perp = [math.exp(min(l,10)) for l in losses_log]
    ax2.plot(steps_log, perp, 'g-', lw=1.5)
    ax2.set_title('Perplexity (log scale)')
    ax2.set_xlabel('Steps'); ax2.set_ylabel('Perplexity')
    ax2.set_yscale('log'); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/loss_curve.png', dpi=150)
    plt.show()
    print(f'Start PPL : {math.exp(losses_log[0]):.1f}')
    print(f'Final PPL : {math.exp(min(losses_log[-1],10)):.1f}')

In [ ]:
# STEP 7 — Evaluate + Generate
os.chdir(WORK_DIR)
TRAIN_FILE = '/kaggle/working/dataset.txt'

ckpts = sorted([f for f in glob.glob('*.bin') if 'vocab' not in f])
print('Checkpoints:', ckpts)

if ckpts:
    latest = ckpts[-1]
    print(f'\n📊 Evaluating: {latest}')
    os.system(f'./build/logos --eval {TRAIN_FILE} {latest}')
    print('\n=== TEXT GENERATION ===')
    for p in ['Once upon a time', 'The little girl', 'Tom liked to']:
        print(f"\nPrompt: '{p}'")
        os.system(f'./build/logos --generate {latest} "{p}"')
else:
    print('⚠️  Pehle Step 5 chalao')

In [ ]:
# STEP 8 — Save Output
os.chdir(WORK_DIR)
OUT = '/kaggle/working/logos_trained'
os.makedirs(OUT, exist_ok=True)

for f in glob.glob('*.bin'): shutil.copy(f, OUT); print(f'✅ {f}')
shutil.copy('build/logos', OUT); print('✅ logos binary')
for f in ['/kaggle/working/loss_curve.png',
          '/kaggle/working/training_log.txt']:
    if os.path.exists(f): shutil.copy(f, OUT); print(f'✅ {os.path.basename(f)}')

os.system(f'ls -lh {OUT}')
print('\n✅ Output tab se download karo!')